# Proposed validation — review before it runs

**What this measures:** `dice_gap` (segformer3d_dice − segresnet_dice) still measures whether the real `SegFormer3D` module, trained from scratch under an identical fixed-budget protocol as `SegResNet`, reaches segmentation-accuracy parity with an established MONAI net — now on the real cached Task01_BrainTumour 8/4 subset instead of synthetic volumes of the same shape, per the reviewer's explicit instruction.

**Target metric:** `dice_gap`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `eval/eval_segformer3d_brats_parity.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the script at `eval/eval_segformer3d_brats_parity.py`, unchanged. This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_segformer3d_brats_parity.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
#!/usr/bin/env python
"""
Evaluation: does the real `SegFormer3D` module, trained from scratch on the real
Task01_BrainTumour data (via monai.apps.DecathlonDataset), reach segmentation-accuracy
parity with MONAI's own SegResNet under an identical fixed-budget, fixed-seed protocol?

Per user_guidance this now trains on a fixed 8-train/4-validation subset of the REAL
Task01_BrainTumour dataset (downloaded once into the working directory and cached by
monai.apps.DecathlonDataset's own logic), replacing the earlier synthetic-volume design.
Everything else (crop geometry, optimizer, loss, iteration budget, seed) is unchanged.

In [ ]:
`SegFormer3D` is imported defensively: on `dev` (pre-PR) it does not exist, so this
script still trains/evaluates SegResNet alone and reports a (necessarily negative)
dice_gap, rather than crashing -- this is what makes a baseline run possible at all.

Prints exactly one JSON line of metrics and exits 0 on both arms, regardless of
whether the dataset download or training succeeds.
"""

In [ ]:
from __future__ import annotations

import argparse
import json
import os
import sys
import time

In [ ]:
import torch

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, REPO_ROOT)

SEED = 0
CROP = (64, 64, 64)
N_TRAIN, N_VAL = 8, 4
N_ITERS = 200
LR = 1e-4
DATA_ROOT = os.environ.get("MONAI_DATA_DIRECTORY", os.path.join(os.getcwd(), "monai_brats_data"))

In [ ]:
EXISTING_NETS = ["SegResNet", "UNet", "BasicUNet", "DynUNet", "AttentionUnet", "VNet", "SwinUNETR", "UNETR"]

def build_transforms():
    from monai.transforms import (
        CenterSpatialCropd,
        Compose,
        EnsureChannelFirstd,
        EnsureTyped,
        LoadImaged,
        MapTransform,
        NormalizeIntensityd,
        SpatialPadd,
    )

    class ConvertToMultiChannelBratsd(MapTransform):
        """label 1=edema, 2=enhancing tumor, 3=necrotic/non-enhancing core -> (TC, WT, ET)."""

        def __call__(self, data):
            d = dict(data)
            for key in self.keys:
                label = d[key]
                label = label[0] if label.shape[0] == 1 else label
                tc = (label == 2) | (label == 3)
                wt = tc | (label == 1)
                et = label == 2
                d[key] = torch.stack([tc, wt, et], dim=0).float()
            return d

    return Compose(
        [
            LoadImaged(keys=["image", "label"]),
            EnsureChannelFirstd(keys=["image", "label"]),
            EnsureTyped(keys=["image", "label"]),
            SpatialPadd(keys=["image", "label"], spatial_size=CROP),
            CenterSpatialCropd(keys=["image", "label"], roi_size=CROP),
            NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
            ConvertToMultiChannelBratsd(keys=["label"]),
        ]
    )

In [ ]:
def load_subset():
    """Deterministically pull a fixed 8/4 subset from the real, cached Task01_BrainTumour data."""
    from monai.apps import DecathlonDataset

    tf = build_transforms()
    train_ds = DecathlonDataset(
        root_dir=DATA_ROOT,
        task="Task01_BrainTumour",
        section="training",
        transform=tf,
        download=True,
        seed=SEED,
        val_frac=0.2,
        cache_rate=0.0,
        num_workers=0,
        progress=False,
    )
    val_ds = DecathlonDataset(
        root_dir=DATA_ROOT,
        task="Task01_BrainTumour",
        section="validation",
        transform=tf,
        download=False,
        seed=SEED,
        val_frac=0.2,
        cache_rate=0.0,
        num_workers=0,
        progress=False,
    )
    g = torch.Generator().manual_seed(SEED)
    train_idx = torch.randperm(len(train_ds), generator=g)[:N_TRAIN].tolist()
    val_idx = torch.randperm(len(val_ds), generator=g)[:N_VAL].tolist()
    train_items = [train_ds[i] for i in train_idx]
    val_items = [val_ds[i] for i in val_idx]
    return train_items, val_items

In [ ]:
def count_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

def train_and_eval(model, train_items, val_items, device):
    from monai.losses import DiceLoss

    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = DiceLoss(sigmoid=True, squared_pred=True)

    imgs = torch.stack([it["image"] for it in train_items]).to(device)
    labels = torch.stack([it["label"] for it in train_items]).to(device)

    model.train()
    start = time.time()
    for _ in range(N_ITERS):
        opt.zero_grad()
        loss = loss_fn(model(imgs), labels)
        loss.backward()
        opt.step()
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - start

    model.eval()
    val_imgs = torch.stack([it["image"] for it in val_items]).to(device)
    val_labels = torch.stack([it["label"] for it in val_items]).to(device)
    with torch.no_grad():
        pred = (torch.sigmoid(model(val_imgs)) > 0.5).float()
        intersect = (pred * val_labels).sum(dim=(2, 3, 4))
        denom = pred.sum(dim=(2, 3, 4)) + val_labels.sum(dim=(2, 3, 4))
        dice = (2 * intersect / denom.clamp(min=1e-6)).mean().item()
    return dice, count_params(model), elapsed

In [ ]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)
    parser.add_argument("--ref", default=None)
    parser.add_argument("--seed", default=None)
    parser.parse_known_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    try:
        from monai.utils import set_determinism

        set_determinism(seed=SEED)
    except Exception:
        torch.manual_seed(SEED)

    # Guardrail: fraction of MONAI's pre-existing nets still importable after the
    # monai/networks/nets/__init__.py edit this PR makes (shared file, every net).
    nets_module = None
    try:
        import monai.networks.nets as nets_module
    except Exception:
        nets_module = None
    ok = 0
    for name in EXISTING_NETS:
        try:
            if nets_module is not None:
                getattr(nets_module, name)
                ok += 1
        except Exception:
            pass
    existing_nets_import_success_rate = ok / len(EXISTING_NETS)

    # Defensive import of the changed module: absent on `dev`, present on the PR head.
    SegFormer3D = None
    if nets_module is not None:
        try:
            SegFormer3D = getattr(nets_module, "SegFormer3D")
        except Exception:
            SegFormer3D = None

    metrics = {
        "dice_gap": 0.0,
        "existing_nets_import_success_rate": existing_nets_import_success_rate,
        "segformer3d_dice": 0.0,
        "segresnet_dice": 0.0,
        "segformer3d_params": 0,
        "segresnet_params": 0,
        "segformer3d_train_time_s": 0.0,
        "segresnet_train_time_s": 0.0,
    }

    try:
        from monai.networks.nets import SegResNet

        train_items, val_items = load_subset()

        sf_dice = sf_params = sf_time = 0.0
        if SegFormer3D is not None:
            sf_model = SegFormer3D(in_channels=4, out_channels=3)
            sf_dice, sf_params, sf_time = train_and_eval(sf_model, train_items, val_items, device)

        sr_model = SegResNet(spatial_dims=3, in_channels=4, out_channels=3)
        sr_dice, sr_params, sr_time = train_and_eval(sr_model, train_items, val_items, device)

        metrics.update(
            {
                "dice_gap": sf_dice - sr_dice,
                "segformer3d_dice": sf_dice,
                "segresnet_dice": sr_dice,
                "segformer3d_params": int(sf_params),
                "segresnet_params": int(sr_params),
                "segformer3d_train_time_s": sf_time,
                "segresnet_train_time_s": sr_time,
            }
        )
    except Exception as exc:  # dataset download/training must never crash this eval
        print(f"# real-data training path failed, reporting degraded metrics: {exc}", file=sys.stderr)

    print(json.dumps(metrics))

if __name__ == "__main__":
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
loop: {max_iterations: 8, fix_code: true}
benchmarks:
  - name: segformer3d-brats-architecture-parity
    suite: "eval/eval_segformer3d_brats_parity.py"
    scorer: dice_gap
    baseline: dev
    metrics:
      - name: dice_gap
        role: target
        direction: max
        threshold: -0.05
      - name: existing_nets_import_success_rate
        role: guardrail
        direction: max
        threshold: 1.0
      - name: segformer3d_dice
        role: cost
        direction: max
        threshold: 0.0
      - name: segresnet_dice
        role: cost
        direction: max
        threshold: 0.0
      - name: segformer3d_params
        role: cost
        direction: min
        threshold: 10000000
      - name: segresnet_params
        role: cost
        direction: min
        threshold: 10000000
      - name: segformer3d_train_time_s
        role: cost
        direction: min
        threshold: 1800
      - name: segresnet_train_time_s
        role: cost
        direction: min
        threshold: 1800
    policy: {guardrail_veto: true}
held_constant:
  - "same fixed 8/4 subset of the real Task01_BrainTumour training list: deterministically selected with one fixed seed for both models, identical inputs for both arms"
  - "same crop geometry: (64, 64, 64) voxels, same seed, same crop/resize pipeline applied to the real Task01_BrainTumour volumes for both models"
  - "same optimizer, learning rate and loss: Adam(lr=1e-4) with DiceLoss(sigmoid=True, squared_pred=True), identical for both models"
  - "same fixed step budget: 200 from-scratch training iterations for both models, no pretrained weights for either"
  - "same torch/numpy/monai seed across both models via monai.utils.set_determinism, single seed only, no multi-seed averaging"
  - "Task01_BrainTumour is fetched exactly once via monai.apps.DecathlonDataset(root_dir=<working dir>, task='Task01_BrainTumour', download=True) and cached in the working directory; repeat runs reuse the cache rather than re-downloading"
avoid:
  - "this is a fixed-budget, from-scratch comparison on a fixed real 8/4 Task01_BrainTumour subset -- a parity signal, not a reproduction of either paper's full training protocol or fully converged accuracy on all of BraTS"
  - "the multi-gigabyte Task01_BrainTumour download is now exercised per user_guidance; it is fetched once into the working directory and cached, so repeat invocations of this script must detect the existing cache and skip re-downloading to stay practical"
  - "no two-arm delta is used for the target: the dev baseline cannot import SegFormer3D at all, so both models are trained and scored from the feature arm and the gap is read against a fixed bound"
  - "train time and parameter counts are reported only as cost references, never as a pass/fail gate on model quality"
  - "no invented paper Dice numbers: only Dice computed by this script's own from-scratch training run on the real cached subset is reported"
  - "single seed only, as instructed: no seed-to-seed variance estimate exists for dice_gap, so the fixed -0.05 tolerance band remains a fixed bound rather than a noise-derived one"
compute:
  tier: gpu
  timeout_s: 5400
provenance:
  dice_gap: "user_guidance (fixed-budget from-scratch Dice comparison on the real Task01_BrainTumour 8/4 subset, no two-arm delta)"
  existing_nets_import_success_rate: "inferred -- regression guardrail for the monai/networks/nets/__init__.py edit this PR makes, since that file is shared by every existing net"
  segformer3d_dice: "user_guidance (mean validation Dice per model)"
  segresnet_dice: "user_guidance (mean validation Dice per model)"
  segformer3d_params: "user_guidance (params reported as a cost reference only)"
  segresnet_params: "user_guidance (params reported as a cost reference only)"
  segformer3d_train_time_s: "user_guidance (train time reported as a cost reference only)"
  segresnet_train_time_s: "user_guidance (train time reported as a cost reference only)"
  held_constant: "user_guidance (same fixed 8/4 subset and seed, downloaded once via monai.apps.DecathlonDataset) refined by protocol_doc:monai/networks/nets/segformer3d.py for the channel/crop geometry"
  compute: "inferred -- timeout_s raised from 3600 to 5400 to cover the one-time Task01_BrainTumour download alongside the unchanged 200-iteration x2-model training budget"
  suite: "synthesized (R1 maturity repo: tests + CI only, no BraTS benchmark harness exists to run (a) against); now loads the real cached Task01_BrainTumour subset via monai.apps.DecathlonDataset per user_guidance instead of synthetic volumes"
```